In [2]:
import json
import random
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import Dataset, DataLoader


board_size = 15 # 15 = mid


import os
import glob


board_size = 15  # Board size for three-player Hex

# Load all training data from JSON files in a folder
def load_training_data_from_folder(folder_path):
    all_data = []
    json_files = glob.glob(os.path.join(folder_path, "*.json"))  # Get all JSON files
    for file_path in json_files:
        with open(file_path, "r") as file:
            data = json.load(file)
            all_data.extend(data)  # Merge data from all files
    random.shuffle(all_data)        
    return all_data

class HexDataset(Dataset):
    def __init__(self, folder_path):
        self.data = load_training_data_from_folder(folder_path)
        self.board_size = board_size

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]

        # Convert list to numpy array and reshape to (3, board_size, board_size)
        player1_board = np.array(sample["Player1Board"]).reshape(1, self.board_size, self.board_size)
        player2_board = np.array(sample["Player2Board"]).reshape(1, self.board_size, self.board_size)
        player3_board = np.array(sample["Player3Board"]).reshape(1, self.board_size, self.board_size)

        # Stack into a (3, board_size, board_size) tensor
        board_tensor = torch.tensor(np.vstack([player1_board, player2_board, player3_board]), dtype=torch.float32)

        # Target value (scalar)
        root_value = torch.tensor([sample["RootValue"]], dtype=torch.float32)

        # Target policy (1D tensor of board_size * board_size)
        policy = torch.tensor(sample["MoveProbabilities"], dtype=torch.float32)

        return board_tensor, policy, root_value


In [3]:
class ThreePlayerHexNN(nn.Module):
    def __init__(self):
        super(ThreePlayerHexNN, self).__init__()
        self.board_size = board_size
        input_channels = 3  # One binary 2D array per player

        # Initial convolution layer
        self.conv1 = nn.Conv2d(input_channels, 256, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(256)

        # Residual blocks
        self.res_block1 = self._residual_block(256)
        self.res_block2 = self._residual_block(256)
        self.res_block3 = self._residual_block(256)
        self.res_block4 = self._residual_block(256)
        self.res_block5 = self._residual_block(256)

        # Policy Head
        self.policy_head = nn.Sequential(
            nn.Conv2d(256, 2, kernel_size=1),
            nn.BatchNorm2d(2),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(2 * board_size * board_size, board_size * board_size),
            nn.Softmax(dim=-1)
        )

        # Value Head
        self.value_head = nn.Sequential(
            nn.Conv2d(256, 1, kernel_size=1),
            nn.BatchNorm2d(1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(board_size * board_size, 256),
            nn.ReLU(),
            nn.Linear(256, 1),
            nn.Tanh()
        )

    def _residual_block(self, channels):
        return nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(channels),
            nn.ReLU(),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(channels),
            nn.ReLU()
        )

    def forward(self, x):
        x = torch.relu(self.bn1(self.conv1(x)))
        x = self.res_block1(x) + x
        x = self.res_block2(x) + x
        x = self.res_block3(x) + x
        x = self.res_block4(x) + x
        x = self.res_block5(x) + x
        
        policy = self.policy_head(x)
        value = self.value_head(x)
        return policy, value


In [4]:
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
print("CUDA Version:", torch.version.cuda)
print("CuDNN Version:", torch.backends.cudnn.version())
print("GPU Count:", torch.cuda.device_count())
print("Current GPU:", torch.cuda.current_device() if torch.cuda.is_available() else "No GPU detected")
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")


PyTorch Version: 2.6.0+cpu
CUDA Available: False
CUDA Version: None
CuDNN Version: None
GPU Count: 0
Current GPU: No GPU detected
GPU Name: No GPU detected


In [5]:
# Training configuration
batch_size = 80
num_epochs = 200
learning_rate = 0.022
data_folder = "V:/MFF/bakalarka/training_data/"  # Path to the folder with JSON files

# Create dataset and dataloader
dataset = HexDataset(data_folder)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Initialize model, optimizer, and loss functions
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
model = ThreePlayerHexNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

policy_loss_fn = nn.CrossEntropyLoss()  # Policy loss
value_loss_fn = nn.MSELoss()            # Value loss

Using device: cpu


In [5]:


# Training loop
for epoch in range(num_epochs):
    total_loss = 0
    for boards, policies, values in dataloader:
        boards, policies, values = boards.to(device), policies.to(device), values.to(device)

        optimizer.zero_grad()

        # Forward pass
        predicted_policies, predicted_values = model(boards)

        # Compute losses
        policy_loss = policy_loss_fn(predicted_policies, policies)
        value_loss = value_loss_fn(predicted_values.squeeze(), values)

        loss = policy_loss + value_loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss / len(dataloader):.4f}")

# Save the trained model
torch.save(model.state_dict(), "trained_three_player_hex.pth")
print("Model saved successfully!")


C:\Users\Mazi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\nn\modules\loss.py:608: UserWarning: Using a target size (torch.Size([80, 1])) that is different to the input size (torch.Size([80])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
C:\Users\Mazi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\nn\modules\loss.py:608: UserWarning: Using a target size (torch.Size([78, 1])) that is different to the input size (torch.Size([78])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch [1/200], Loss: 6.6424
Epoch [2/200], Loss: 6.6479
Epoch [3/200], Loss: 6.6470
Epoch [4/200], Loss: 6.6462
Epoch [5/200], Loss: 6.6460
Epoch [6/200], Loss: 6.6452
Epoch [7/200], Loss: 6.6446
Epoch [8/200], Loss: 6.6445
Epoch [9/200], Loss: 6.6443
Epoch [10/200], Loss: 6.6437
Epoch [11/200], Loss: 6.6435
Epoch [12/200], Loss: 6.6431
Epoch [13/200], Loss: 6.6428
Epoch [14/200], Loss: 6.6424
Epoch [15/200], Loss: 6.6422
Epoch [16/200], Loss: 6.6422
Epoch [17/200], Loss: 6.6417
Epoch [18/200], Loss: 6.6416
Epoch [19/200], Loss: 6.6417
Epoch [20/200], Loss: 6.6414
Epoch [21/200], Loss: 6.6413
Epoch [22/200], Loss: 6.6410
Epoch [23/200], Loss: 6.6411
Epoch [24/200], Loss: 6.6410
Epoch [25/200], Loss: 6.6408
Epoch [26/200], Loss: 6.6408
Epoch [27/200], Loss: 6.6408
Epoch [28/200], Loss: 6.6408
Epoch [29/200], Loss: 6.6406
Epoch [30/200], Loss: 6.6407
Epoch [31/200], Loss: 6.6406
Epoch [32/200], Loss: 6.6405
Epoch [33/200], Loss: 6.6403
Epoch [34/200], Loss: 6.6402
Epoch [35/200], Loss: 6

In [6]:
model.load_state_dict(torch.load("trained_three_player_hex.pth"))
model.eval()


RuntimeError: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.

In [1]:
# save the model as .onnx

import torch
import torch.nn as nn
import torch.optim as optim
import onnx
board_size = 15  # Board size for three-player Hex
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ThreePlayerHexNN().to(device)

dummy_input = torch.randn(1, 3, board_size, board_size).to(device)
torch.onnx.export(
    model,  # Model
    dummy_input,  # Example input tensor
    "three_player_hex2.onnx",  # Output ONNX file name
    input_names=["board"],  # Name of input tensor
    output_names=["policy", "value"],  # Names of output tensors
    opset_version=12,  # ONNX opset version (adjust if needed)
    dynamic_axes={"board": {0: "batch_size"}, "policy": {0: "batch_size"}, "value": {0: "batch_size"}},  # Dynamic batch size
    verbose=True  # Print ONNX export details
)

print("Model exported successfully as three_player_hex.onnx!")
print("Model exported successfully!")


C:\Users\Mazi\AppData\Roaming\Python\Python312\site-packages\torch\utils\_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


NameError: name 'ThreePlayerHexNN' is not defined

In [ ]:
onnx_model = onnx.load("three_player_hex.onnx")
onnx.checker.check_model(onnx_model)  # Raise error if the model is invalid

print("ONNX model is valid!")


In [ ]:
# CHECK THE TRAINING DATA IS OK


import json
import numpy as np

with open(data_path, "r") as file:
    data = json.load(file)

# Function to dynamically calculate board size and visualize a single sample
def visualize_training_sample(sample):
    """ Dynamically computes board size and prints merged board representation. """

    # Calculate board size dynamically (since it's always square)
    total_cells = len(sample["Player1Board"])
    size = int(np.sqrt(total_cells))  # Assumes square board

    # Convert 1D lists back to 2D numpy arrays
    p1_board = np.array(sample["Player1Board"]).reshape((size, size))
    p2_board = np.array(sample["Player2Board"]).reshape((size, size))
    p3_board = np.array(sample["Player3Board"]).reshape((size, size))

    # Merge all boards into a single visual representation
    merged_board = np.zeros((size, size), dtype=int)
    merged_board[p1_board == 1] = 1  # Mark player 1 cells
    merged_board[p2_board == 1] = 2  # Mark player 2 cells
    merged_board[p3_board == 1] = 3  # Mark player 3 cells

    # Print the boards
    print(f"\n--- Player 1 Board ({size}x{size}) ---")
    print(p1_board)
    print(f"\n--- Player 2 Board ({size}x{size}) ---")
    print(p2_board)
    print(f"\n--- Player 3 Board ({size}x{size}) ---")
    print(p3_board)
    print(f"\n--- Merged Board ({size}x{size}) ---")
    print(merged_board)

    # Print additional metadata
    print("\nRoot Value:", sample["RootValue"])
    print("Move Probabilities:")
    print(np.array(sample["MoveProbabilities"]).reshape((size, size)))

# Show the first training sample
if data:
    print("\nLoaded Training Samples:", len(data))
    visualize_training_sample(data[0])  # Print the first sample
else:
    print("No training samples found in the file.")
